In [ ]:
#OOP & Design Patterns cho DE
#Classes cơ bản và Dataclasses
#class truyền thống

class DatabaseConfig:
    def __init__(self, host: str, port: int, database: str, user: str, password):
        self.host = host
        self.port = port
        self.database = database
        self.user = user
        self._password = password #_tiền tố - quy định trong nội bộ không nên truy cập từ bên ngoài
    
    @property                       # biến method bên dưới thành thuộc tính — gọi không cần ()
    def connection_string(self) -> str:     # -> str nghĩa là hàm này trả về kiểu str
        '''tạo connection string (computed property)'''
        # ghép các thông tin thành chuỗi kết nối PostgreSQL, ẩn password bằng ***
        return f"postgresql://{self.user}:***@{self.host}:{self.port}/{self.database}"
 
    def __repr__(self) -> str:      # Python tự gọi khi bạn print() hoặc gõ object trong terminal
        # trả về chuỗi mô tả object để dễ debug — thay vì thấy địa chỉ bộ nhớ
        return f"DatabaseConfig({self.host}:{self.port}/{self.database})"
 
    def __eq__(self, other) -> bool:    # Python tự gọi khi bạn dùng == để so sánh 2 object
        # 2 config được coi là "bằng nhau" nếu cùng host, port và database
        # bỏ qua user và password khi so sánh
        return (self.host == other.host and         # so sánh host
                self.port == other.port and         # so sánh port
                self.database == other.database)    # so sánh  database
        

    

In [ ]:
#@dataclass - cách python hiện đại từ phiên bản 3.7+
#tự động tạo __init__, __repr__, __eq__

from dataclasses import dataclass, field

@dataclass
class OrderRecord:
    order_id: int
    customer_id: int
    amount: float
    status: str = 'pending' #kiểu giá trị mặc định
    items: list = field(default_factory=list) # có thể thay đổi mặc định
    
# tạo instance - giống dict nhưng có tyoe hints

order = OrderRecord(order_id=1, customer_id=100, amount=500_000)
print(order) 



OrderRecord(order_id=1, customer_id=100, amount=500000, status='pending', items=[])


In [7]:
#@dataclass(froxzen = True) - 
@dataclass(frozen = True) 
# frozen=True = object không thể thay đổi sau khi tạo (immutable)
# tự động tạo __init__, __repr__, __eq__ dựa trên các field bên dưới
class PartitionKey:
    year: int
    month: int
    day: int
    
    @property # biến method thành thuộc tính — gọi không cần ()
    def path(self) -> str:
        return f"{self.year}-{self.month}-{self.day:02d}"  # ghép thành chuỗi dạng "2026-1-09"
    
key = PartitionKey(2006, 19, 9)    # tạo object — frozen=True nên key.year = 999 sau này sẽ bị lỗi
print(key.path) # gọi property path — in ra "2026-19-09"

2006-19-09


In [ ]:
# dâtclass với __post_init__ - validation

@dataclass 
class product: #u như kỉ
    product_id: int
    name: str
    price: float
    category: str
    
    def __post_init__(self): #chạy tự động sau khi __init__ chạy và dùng để validate dữ liệu đầu vào cho đúng
        if self.price < 0:   
            raise ValueError(f"giá tiền ko thể âm {self.price}")
        if not self.name.strip():
            raise ValueError(f"tên sản phẩm ko đc viết khoản trắng")
        self.name = self.name.strip().title() #chinh sửa dữ liệu name lại thành xóa khoản trắng và viết hoa chữ cái đầu

san_pham = product(1, "         bàn phím cơ    ", 100_000, "siêu mượt và êm")
print(san_pham)

product(product_id=1, name='Bàn Phím Cơ', price=100000, category='siêu mượt và êm')
